# Gold 01 — Build Gold Layer

Reads from Silver Delta tables, builds the track spine dimension,
maps all telemetry to the spine, and writes three Gold tables ready
for Power BI.

**Prerequisites:**
- `silver_replay_header` and `silver_replay_telemetry` Delta tables exist

**Output Gold tables:**
- `gold_replay_header` — Replay metadata (copy of Silver with ingestion timestamp)
- `gold_track_spine` — Track dimension — one row per spatial point per map
- `gold_replay_telemetry` — All telemetry mapped to spine with enriched columns

Note: The `spark` variable is pre-defined in Fabric Notebooks.

## Parameters

In [ ]:
# Silver tables
SILVER_HEADER_TABLE      = "silver_replay_header"
SILVER_TELEMETRY_TABLE   = "silver_replay_telemetry"
SILVER_CHECKPOINTS_TABLE = "silver_replay_checkpoints"

# Gold tables
GOLD_HEADER_TABLE        = "gold_replay_header"
GOLD_TRACK_SPINE_TABLE   = "gold_track_spine"
GOLD_TELEMETRY_TABLE     = "gold_replay_telemetry"

# Source filter for track spine
SPINE_SOURCE = "player"

# Override spine replay selection per map.
# List replay_id values to use for the spine on their respective maps.
# If a map has no override here, the SLOWEST replay (by race_time_ms) is used as fallback.
SPINE_REPLAY_IDS = [
    # "abc123def456...",   # replay_id for a specific map
]


## Read Silver Tables

In [ ]:
from pyspark.sql.functions import (
    col, lit, row_number, min as spark_min, sqrt, pow as spark_pow,
    monotonically_increasing_id, current_timestamp, sum as spark_sum,
    when, coalesce, split, size, broadcast, collect_list, sort_array
)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, FloatType

df_header      = spark.table(SILVER_HEADER_TABLE)
df_telemetry   = spark.table(SILVER_TELEMETRY_TABLE)
df_checkpoints = spark.table(SILVER_CHECKPOINTS_TABLE)

## Gold Replay Header

In [ ]:
df_gold_header = df_header.withColumn("ingested_to_gold_at", current_timestamp())

df_gold_header.write.format("delta").mode("overwrite").saveAsTable(GOLD_HEADER_TABLE)

print(f"\u2713 Wrote {df_gold_header.count()} rows to gold_replay_header")

## Enrich Telemetry with Checkpoint Section

Add `checkpoint_section` to each telemetry row by joining with `silver_replay_checkpoints`.
Section N = driving towards checkpoint N (section 1 from start until CP1 is crossed, etc.).

Also adds:
- `distance_per_sample` = speed × 0.05 (speed × 50ms in seconds)
- `cumulative_distance` = running total of `distance_per_sample` per replay


In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType as _Int

# Build sorted array of cumulative CP times per replay
df_cp_arrays = (
    df_checkpoints
    .groupBy("replay_id")
    .agg(sort_array(collect_list("checkpoint_time_ms")).alias("cp_times"))
)

df_tel_cp = df_telemetry.join(broadcast(df_cp_arrays), on="replay_id", how="left")

def _get_section(time_ms, cp_times):
    if cp_times is None or len(cp_times) == 0:
        return 1
    for i, cp in enumerate(cp_times):
        if time_ms <= cp:
            return i + 1
    return len(cp_times) + 1

get_section_udf = udf(_get_section, _Int())

df_enriched = (
    df_tel_cp
    .withColumn("checkpoint_section", get_section_udf(col("time_ms"), col("cp_times")))
    .drop("cp_times")
)

w_cumulative = Window.partitionBy("replay_id").orderBy("time_ms").rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)
df_enriched_telemetry = (
    df_enriched
    .withColumn("distance_per_sample", col("speed").cast(FloatType()) * lit(0.05))
    .withColumn("cumulative_distance",  spark_sum("distance_per_sample").over(w_cumulative))
)

# Fix gear values: raw encoding gives 1.0/1.8/2.6/3.4/4.2 instead of 1/2/3/4/5
df_enriched_telemetry = df_enriched_telemetry.withColumn(
    "gear",
    when(col("gear") < 0.5, 0)
    .when(col("gear") < 1.5, 1)
    .when(col("gear") < 2.3, 2)
    .when(col("gear") < 3.1, 3)
    .when(col("gear") < 3.9, 4)
    .otherwise(5)
    .cast("int")
)

print("\u2713 Checkpoint sections assigned from silver_replay_checkpoints")
print("\u2713 Gear values remapped to 0\u20135")
print("\u2713 distance_per_sample and cumulative_distance added")

## Build Track Spine

Selects one replay per map to act as the spatial spine.

- If a `replay_id` is listed in `SPINE_REPLAY_IDS`, that replay is used for its map.
- Otherwise, the **slowest** replay (by `race_time_ms`) is used as fallback.

Rationale: slower replays tend to follow a smoother, wider line which makes a better spine.

In [ ]:
from pyspark.sql.functions import desc

# Replays explicitly chosen for the spine
df_override = (
    df_header
    .filter(col("replay_id").isin(SPINE_REPLAY_IDS))
    .select("replay_id", "map_uid")
)

# For maps without an override, pick the slowest replay
override_map_uids = [row.map_uid for row in df_override.select("map_uid").collect()]

w_slowest = Window.partitionBy("map_uid").orderBy(col("race_time_ms").desc())

df_fallback = (
    df_header
    .filter(col("source") == SPINE_SOURCE)
    .filter(~col("map_uid").isin(override_map_uids))
    .withColumn("rank", row_number().over(w_slowest))
    .filter(col("rank") == 1)
    .select("replay_id", "map_uid")
)

# Combine: overrides + fallback
df_spine_replays = df_override.unionByName(df_fallback)


df_spine_telemetry = (
    df_enriched_telemetry
    .join(df_spine_replays, on="replay_id", how="inner")
    .select("map_uid", "checkpoint_section", "x", "y", "z", "time_ms")
    .orderBy("map_uid", "time_ms")
)

df_track_spine = (
    df_spine_telemetry
    .withColumn("track_point_id", monotonically_increasing_id())
    .select("track_point_id", "map_uid", "checkpoint_section", "x", "y", "z")
)

df_track_spine.write.format("delta").mode("overwrite").saveAsTable(GOLD_TRACK_SPINE_TABLE)

print(f"\u2713 Wrote {df_track_spine.count()} track spine points")

## Map Telemetry to Track Spine

For each telemetry row, find the nearest track spine point. Only compare within the same
`map_uid` AND `checkpoint_section` to keep the join manageable and the comparisons meaningful.

Distance formula: `D = sqrt((x1-x2)² + (y1-y2)² + (z1-z2)²)`

In [ ]:
df_telemetry_with_map = (
    df_enriched_telemetry
    .join(df_header.select("replay_id", "map_uid"), on="replay_id", how="inner")
)

df_spine_renamed = (
    df_track_spine
    .withColumnRenamed("x", "spine_x")
    .withColumnRenamed("y", "spine_y")
    .withColumnRenamed("z", "spine_z")
)

df_crossed = df_telemetry_with_map.join(
    broadcast(df_spine_renamed),
    on=["map_uid", "checkpoint_section"],
    how="inner"
)

df_with_dist = df_crossed.withColumn(
    "dist",
    sqrt(
        spark_pow(col("x") - col("spine_x"), 2) +
        spark_pow(col("y") - col("spine_y"), 2) +
        spark_pow(col("z") - col("spine_z"), 2)
    )
)

w_min_dist = Window.partitionBy("replay_id", "time_ms").orderBy(col("dist").asc())

df_mapped = (
    df_with_dist
    .withColumn("dist_rank", row_number().over(w_min_dist))
    .filter(col("dist_rank") == 1)
    .drop("dist_rank", "spine_x", "spine_y", "spine_z")
    .withColumnRenamed("dist", "distance_to_spine")
)


## Write Gold Telemetry

In [ ]:
df_gold_telemetry = df_mapped.select(
    "replay_id",
    "map_uid",
    "track_point_id",
    "checkpoint_section",
    "distance_to_spine",
    "distance_per_sample",
    "cumulative_distance",
    "time_ms", "time_s",
    "x", "y", "z",
    "speed", "side_speed",
    "vel_x", "vel_y", "vel_z",
    "pitch_deg", "yaw_deg", "roll_deg",
    "steer", "gas", "brake",
    "rpm", "gear",
    "is_turbo", "turbo_time",
    "is_ground_contact", "is_top_contact",
    "fl_dampen", "fr_dampen", "rr_dampen", "rl_dampen",
    "fl_ice", "fr_ice", "rr_ice", "rl_ice",
    "fl_dirt", "fr_dirt", "rr_dirt", "rl_dirt",
    "fl_slip", "fr_slip", "rr_slip", "rl_slip",
    "fl_ground_mat", "fr_ground_mat", "rr_ground_mat", "rl_ground_mat",
    "fl_wheel_rot", "fr_wheel_rot", "rr_wheel_rot", "rl_wheel_rot",
    "reactor_state", "reactor_boost",
    "reactor_pedal", "reactor_steer",
    "wetness", "sim_time_coef",
)

df_gold_telemetry.write.format("delta").mode("overwrite").saveAsTable(GOLD_TELEMETRY_TABLE)

count = df_gold_telemetry.count()
print(f"\u2713 Wrote {count} rows to gold_replay_telemetry")